# Task 5 — Mental Health Chatbot (Gemini 2.5 Flash + Fallback)

## Objective
Build a compassionate mental health companion chatbot that:
- Responds gently and validates feelings
- Avoids clinical advice and diagnosis
- Suggests grounding / breathing exercises
- Uses a crisis filter to provide help resources

## Approach
- Primary model: **Gemini 2.5 Flash** via `google-generativeai`
- Fallback: `microsoft/DialoGPT-medium` via `transformers` if Gemini fails
- Keep short conversation memory (last 3 turns)
- UI: Gradio chat interface (works in notebooks), with a CLI fallback

## Final Summary
This notebook demonstrates advanced prompting (no fine-tuning) with safety routing for crisis language, plus a free local fallback model option.

In [ ]:
# pip install google-generativeai gradio transformers torch

import os
import re
from typing import List, Tuple, Optional

try:
    import google.generativeai as genai
except Exception as e:
    genai = None
    print('Failed to import google-generativeai. Install it with pip. Error:', e)

try:
    import gradio as gr
except Exception as e:
    gr = None
    print('Failed to import gradio. Install it with pip. Error:', e)

# Optional fallback dependencies
try:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    import torch
except Exception as e:
    AutoTokenizer = None
    AutoModelForCausalLM = None
    torch = None
    print('Transformers/torch not available for fallback. Install with pip if needed. Error:', e)


In [ ]:
SYSTEM_PROMPT = (
    'You are a compassionate mental health companion. Respond gently, validate feelings, '
    'avoid clinical advice. Suggest breathing exercises, grounding techniques, or talking '
    'to a professional if needed.'
)

CRISIS_PATTERNS = [
    r'\b(suicide|kill myself|end my life|self harm|self-harm|hurt myself)\b',
    r'\b(i don\'?t want to live|i want to die)\b',
]

def is_crisis(text: str) -> bool:
    t = (text or '').lower()
    return any(re.search(p, t) for p in CRISIS_PATTERNS)

def crisis_message() -> str:
    # Keep general and non-region-specific; encourage local emergency contacts
    return (
        'I\'m really sorry you\'re feeling this way. You deserve support right now. '
        'If you\'re in immediate danger or might act on these thoughts, please call your local emergency number right now. '
        'If you\'re in the U.S. or Canada, you can call or text **988** (Suicide & Crisis Lifeline). '
        'If you\'re elsewhere, consider contacting your local crisis hotline or a trusted person, and seek professional help.'
    )


In [ ]:
def build_gemini_model() -> Optional[object]:
    if genai is None:
        return None
    api_key = os.getenv('GOOGLE_API_KEY')
    if not api_key:
        print('Missing GOOGLE_API_KEY env var. Set it to your free Google AI Studio key.')
        return None
    try:
        genai.configure(api_key=api_key)
        return genai.GenerativeModel(
            model_name='gemini-2.5-flash',
            system_instruction=SYSTEM_PROMPT,
        )
    except Exception as e:
        print('Failed to configure Gemini client:', e)
        return None

gemini_model = build_gemini_model()
gemini_model

In [ ]:
_fallback_tokenizer = None
_fallback_model = None

def load_fallback_model():
    global _fallback_tokenizer, _fallback_model
    if _fallback_model is not None and _fallback_tokenizer is not None:
        return _fallback_tokenizer, _fallback_model

    if AutoTokenizer is None or AutoModelForCausalLM is None:
        raise RuntimeError('Fallback requires transformers+torch. Install them to enable DialoGPT fallback.')

    name = 'microsoft/DialoGPT-medium'
    _fallback_tokenizer = AutoTokenizer.from_pretrained(name)
    _fallback_model = AutoModelForCausalLM.from_pretrained(name)
    return _fallback_tokenizer, _fallback_model

def dialo_gpt_reply(prompt: str) -> str:
    tok, mdl = load_fallback_model()
    inputs = tok.encode(prompt + tok.eos_token, return_tensors='pt')
    with torch.no_grad():
        output_ids = mdl.generate(
            inputs,
            max_new_tokens=120,
            do_sample=True,
            top_p=0.92,
            temperature=0.8,
            pad_token_id=tok.eos_token_id,
        )
    text = tok.decode(output_ids[0], skip_special_tokens=True)
    # Return only the assistant continuation when possible
    return text[len(prompt):].strip() or text.strip()


## Conversation policy
- The bot keeps the last **3 turns** of context.
- If crisis language is detected, it returns a crisis-support message instead of continuing normal chat.

### Example empathetic exchanges
- User: "I feel overwhelmed and anxious lately."
  Bot: "That sounds really heavy to carry. Want to tell me what has been most stressful? We can also try a short breathing exercise together."
- User: "I can't stop worrying at night."
  Bot: "Nights can make worries feel louder. A grounding trick: name 5 things you can see, 4 you can feel, 3 you can hear..."

In [ ]:
def format_context(history: List[Tuple[str, str]]) -> str:
    # Keep last 3 turns (user, bot)
    recent = history[-3:]
    lines = []
    for u, b in recent:
        lines.append(f'User: {u}')
        lines.append(f'Assistant: {b}')
    return '\n'.join(lines).strip()

def gemini_reply(user_text: str, history: List[Tuple[str, str]]) -> str:
    if gemini_model is None:
        raise RuntimeError('Gemini is not configured (missing API key or package).')

    context = format_context(history)
    prompt = (
        f'{context}\n' if context else ''
    ) + f'User: {user_text}\nAssistant:'

    resp = gemini_model.generate_content(prompt)
    return (resp.text or '').strip() or 'No response text received.'

def chat(user_text: str, history: List[Tuple[str, str]]) -> str:
    if is_crisis(user_text):
        return crisis_message()

    # Primary: Gemini
    try:
        return gemini_reply(user_text, history)
    except Exception as e:
        # Fallback: DialoGPT (may require large downloads)
        try:
            ctx = format_context(history)
            prompt = (ctx + '\n' if ctx else '') + f'User: {user_text}\nAssistant:'
            return dialo_gpt_reply(prompt)
        except Exception as e2:
            return (
                'I had trouble reaching the online model and the local fallback is not available. '
                f'Gemini error: {e}. Fallback error: {e2}'
            )


In [ ]:
# Gradio interface (recommended in notebooks)
def gradio_respond(message, chat_history):
    chat_history = chat_history or []
    reply = chat(message, chat_history)
    chat_history.append((message, reply))
    return chat_history, chat_history

if gr is not None:
    with gr.Blocks() as demo:
        gr.Markdown('# Mental Health Chatbot (Gemini 2.5 Flash)')
        chatbot = gr.Chatbot(height=350)
        msg = gr.Textbox(label='Type your message')
        state = gr.State([])
        btn = gr.Button('Send')

        btn.click(gradio_respond, inputs=[msg, state], outputs=[chatbot, state])
        msg.submit(gradio_respond, inputs=[msg, state], outputs=[chatbot, state])

    # Uncomment to launch locally:
    # demo.launch()
else:
    print('Gradio not installed; use the CLI loop below.')


In [ ]:
def run_cli():
    print('Mental Health Chatbot. Type `quit` to exit.')
    history = []
    while True:
        user_text = input('You: ').strip()
        if not user_text:
            continue
        if user_text.lower() == 'quit':
            print('Bye!')
            break
        reply = chat(user_text, history)
        history.append((user_text, reply))
        print('Bot:', reply)

# Uncomment to run interactively in a local terminal/Jupyter:
# run_cli()
